In [58]:
# ============================================================
# LLM-as-a-Judge Evaluation for FinOCRBench
# ============================================================
# This notebook evaluates OCR model output quality at the entity level.
#
# Pipeline overview:
#   1. For each sample, load the human-annotated gold HTML (with custom tags like
#      <number>, <monetaryunit>, etc.) and the OCR model's predicted HTML.
#   2. Deterministically extract all tagged entities from the gold HTML using
#      BeautifulSoup — no LLM involved at this stage.
#   3. Send the extracted entities + both HTMLs to GPT-4o as a judge. The judge
#      determines for each entity whether the model found it (found_in_prediction)
#      and whether it got it exactly right (correct).
#   4. Post-process the judge's response: overwrite entity counts using the
#      authoritative programmatic values, and apply a deterministic override
#      (if matched_text == value, force correct=True regardless of LLM judgment).
#   5. Aggregate results with evaluate_performance() to compute precision-style
#      accuracy: correct / found_in_prediction (not correct / total_gold).
#
# Entity types tracked: Number, Temporal, Monetary Unit, Reporting Entity,
#                       Financial Concepts
# ============================================================

import os
import json
import re
from typing import Dict, Any, List
from openai import OpenAI
from tqdm.std import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv

In [ ]:
# ── Configuration ────────────────────────────────────────────
# Paths are relative to the FinOCRBench/code/ working directory.

# Directory containing the OCR model's predicted outputs (pred_0.txt, pred_1.txt, ...)
# model_html = "results/MM_2026_Results/paddleocrv5_zero-shot"
model_html = "path/to/model/predictions"  # <-- UPDATE THIS PATH TO YOUR OCR PREDICTIONS

# Directory containing the human-annotated gold HTML files (gold_0.txt, gold_1.txt, ...)
# Gold files use custom entity tags: <number>, <temporal>, <monetaryunit>,
# <reportingentity>, <financialconcepts>
# gold_html = "data/gold_annotation_html_859"
gold_html = "path/to/gold/html/files"  # <-- UPDATE THIS PATH TO YOUR GOLD HTML FILES

# Index range of samples to evaluate (start_idx inclusive, end_idx exclusive).
# E.g., start_idx=50, end_idx=100 evaluates samples 50..99.
start_idx = 5
end_idx   = 15

# Model name used for labelling output files
# model_name      = "paddleocrv5_zero-shot"
# model_name_test = "paddleocrv5_zero-shot_test"
model_name      = "your_model_name_here"  # <-- UPDATE THIS TO YOUR MODEL NAME (USED IN OUTPUT FILE NAMES) e.g. "paddleocrv5_zero-shot"
model_name_test =   "your_model_name_here_test"   # <-- UPDATE THIS TO YOUR MODEL NAME + '_test' (USED IN OUTPUT FILE NAMES) e.g. "paddleocrv5_zero-shot_test"

# Directory where per-run JSON results are saved
# case_result = "results/MM_2026_Results/llm_as_judge_output"
case_result = f"results/MM_2026_Results/llm_as_judge_output"  # <-- UPDATE THIS PATH TO YOUR DESIRED OUTPUT DIRECTORY

In [ ]:
from dotenv import load_dotenv
OPEN_AI_KEY = os.getenv("OPENAI_API_KEY")
print("Key loaded:", OPEN_AI_KEY[:10] + "..." if OPEN_AI_KEY else "NOT FOUND")

Key loaded: sk-proj-Kv...


In [61]:
Client = OpenAI(api_key=OPEN_AI_KEY)

In [ ]:
# ── Entity extraction from gold HTML ─────────────────────────
# Gold HTML files contain five custom entity tag types. This mapping translates
# the lowercase HTML tag names to the display names used in evaluation output.
TAG_TO_TYPE = {
    "number":           "Number",
    "temporal":         "Temporal",
    "monetaryunit":     "Monetary Unit",
    "reportingentity":  "Reporting Entity",
    "financialconcepts":"Financial Concepts",
}

# Bare currency symbols that are meaningless on their own as entity values.
# When a <monetaryunit> tag contains only one of these, we combine it with
# the adjacent <number> tag to form a full value like "$65.63".
# Note: scale words like "Dollars in Thousands" are NOT in this set — they are
# already meaningful standalone values and should NOT be combined.
_CURRENCY_SYMBOLS = {"$", "€", "£", "¥", "₹", "₩", "CHF", "USD", "EUR", "GBP"}


def _normalize_hint(text: str) -> str:
    """Collapse artifact spaces introduced by HTML tag boundaries.

    BeautifulSoup's get_text() can insert spaces at tag edges, e.g.,
    <span>$</span><span>66.59</span> → "$ 66.59". This function closes
    those gaps so context hints are comparable to plain prediction text.
    """
    text = re.sub(r'\$\s+(\d)', r'$\1', text)    # "$ 66.59" → "$66.59"
    text = re.sub(r'(\d)\s+%', r'\1%', text)      # "5 %" → "5%"
    text = re.sub(r'\(\s+', '(', text)             # "( loss" → "(loss"
    text = re.sub(r'\s+\)', ')', text)             # "loss )" → "loss)"
    text = re.sub(r' {2,}', ' ', text)             # collapse multiple spaces
    return text.strip()


def _get_context_text(elem, min_len: int = 30) -> str:
    """Return a meaningful text context for a gold entity element.

    Walks up the DOM from elem until it finds an ancestor whose combined
    text is at least min_len characters. This avoids degenerate context like
    "$" (from a <span> containing only the currency symbol) and instead
    returns the surrounding row text like "Class A Common Stock 02/05/2025 P 25,700".
    """
    node = elem.parent
    while node is not None:
        text = node.get_text(" ", strip=True)
        text = _normalize_hint(text)
        if len(text) >= min_len:
            return text
        node = node.parent
    # Fallback: just use the element's own text if no suitable ancestor exists
    return _normalize_hint(elem.get_text(" ", strip=True))


def _get_monetary_value(elem) -> str:
    """Combine a bare currency symbol with its adjacent number to form a full value.

    In the gold HTML, monetary amounts are often annotated as two separate tags:
        <span><monetaryunit>$</monetaryunit></span><span><number>65.63</number></span>
    A bare "$" is not useful as an entity value, so this function walks forward
    through the sibling elements to find the paired <number> tag and returns
    the concatenated string (e.g., "$65.63").

    Checks both direct siblings of elem and siblings of elem's parent span,
    since the number tag may be wrapped in a sibling <span>.

    Note: We guard with `hasattr(sibling, 'name') and sibling.name` to ensure
    we only call BeautifulSoup's .find() on Tag objects. NavigableString also
    inherits from str and has a .find() method, but that returns an int index —
    calling .get_text() on that int would raise AttributeError.
    """
    symbol = elem.get_text(strip=True)

    # First: check direct next siblings of the <monetaryunit> element
    for sibling in elem.next_siblings:
        sibling_text = sibling.get_text(strip=True) if hasattr(sibling, 'get_text') else str(sibling).strip()
        if not sibling_text:
            continue
        if hasattr(sibling, 'name') and sibling.name == 'number':
            # e.g., <monetaryunit>$</monetaryunit><number>65.63</number>
            return symbol + sibling_text
        if hasattr(sibling, 'name') and sibling.name:
            # sibling is a real Tag (not a NavigableString) — safe to call .find()
            num_tag = sibling.find('number')
            if num_tag:
                # e.g., <monetaryunit>$</monetaryunit><span><number>65.63</number></span>
                return symbol + num_tag.get_text(strip=True)
        break  # stop at first non-empty sibling to avoid false positives

    # Second: check next siblings of the parent span
    # (handles: <span><monetaryunit>$</monetaryunit></span><span><number>65.63</number></span>)
    if elem.parent is not None:
        for sibling in elem.parent.next_siblings:
            sibling_text = sibling.get_text(strip=True) if hasattr(sibling, 'get_text') else str(sibling).strip()
            if not sibling_text:
                continue
            if hasattr(sibling, 'name') and sibling.name:
                # sibling is a real Tag — safe to call .find()
                num_tag = sibling.find('number')
                if num_tag:
                    return symbol + num_tag.get_text(strip=True)
            break

    return symbol  # fallback: return just the symbol if no number found nearby


def extract_gold_entities(gold_html: str) -> list:
    """Parse gold HTML and return all annotated entities as a flat list.

    Each entity dict has:
        - "type":         one of the 5 entity type strings (e.g., "Number")
        - "value":        the entity text (e.g., "25,700" or "$65.63")
        - "context_hint": a ~80-char substring of surrounding text for
                          disambiguation when the same value appears multiple times

    Special handling:
        - Bare currency symbols (<monetaryunit>$</monetaryunit>) are combined
          with the adjacent <number> tag to produce a meaningful value.
        - Context hints are built by finding the entity value within its
          nearest ancestor that has ≥30 chars of text, then slicing ±40 chars.
    """
    soup = BeautifulSoup(gold_html, "html.parser")
    entities = []

    for tag, etype in TAG_TO_TYPE.items():
        for elem in soup.find_all(tag):
            value = elem.get_text(strip=True)
            if not value:
                continue

            # For bare currency symbols, combine with the adjacent number tag
            if etype == "Monetary Unit" and value in _CURRENCY_SYMBOLS:
                value = _get_monetary_value(elem)

            # Build context hint: find the value within a meaningful ancestor text
            parent_text = _get_context_text(elem, min_len=30)
            idx = parent_text.find(value)
            if idx >= 0:
                start = max(0, idx - 40)
                end = min(len(parent_text), idx + len(value) + 40)
                context_hint = parent_text[start:end].strip()
            else:
                # Value not found in parent text (can happen after normalization);
                # use the beginning of the parent text as a best-effort hint
                context_hint = parent_text[:80].strip()

            entities.append({"type": etype, "value": value, "context_hint": context_hint})

    return entities


def strip_entity_tags(gold_html: str) -> str:
    """Remove all custom entity annotation tags from gold HTML, keeping inner text.

    The LLM judge receives this stripped version as reference context so it can
    locate where each entity appears in the document structure, without the
    annotation tags confusing its understanding of the HTML.

    Example: <number>25,700</number> → 25,700
    """
    tag_pattern = re.compile(
        r'<(number|temporal|monetaryunit|reportingentity|financialconcepts)>(.*?)</\1>',
        re.IGNORECASE | re.DOTALL
    )
    return tag_pattern.sub(r'\2', gold_html)


def strip_html_tags(text: str) -> str:
    """Convert model-predicted HTML to plain text for the LLM judge.

    LLM-generated predictions (GPT-4o, Llama, Gemini, etc.) output structured
    HTML with <table>, <tr>, <td>, <h1>, etc. OCR tools like PaddleOCR output
    plain text with no tags. Stripping HTML tags normalizes all model outputs
    to plain text so the judge receives consistent input regardless of model type.
    For plain-text predictions (no tags), this is effectively a no-op.

    Uses a space separator in get_text() to avoid merging adjacent cell values
    (e.g., <td>foo</td><td>bar</td> → "foo bar", not "foobar").
    Collapses excessive blank lines to keep the text compact.
    """
    soup = BeautifulSoup(text, "html.parser")
    plain = soup.get_text(separator=" ")
    plain = re.sub(r'\n\s*\n+', '\n', plain)   # collapse blank lines
    plain = re.sub(r' {2,}', ' ', plain)        # collapse multiple spaces
    return plain.strip()

In [63]:
JUDGE_PROMPT_TEMPLATE = """
#Instruction: You are an expert OCR results inspector for financial documents.
You will be given:
1. A pre-extracted list of financially critical entities from the ground truth HTML. Each entity has:
   - "type": one of "Number", "Temporal", "Monetary Unit", "Reporting Entity", "Financial Concepts"
   - "value": the exact text of the entity
   - "context_hint": a short fragment of surrounding text that helps locate it in the document

2. Ground truth HTML (entity tags already stripped, plain structure only) — for reference context.

3. Model-generated HTML that was produced from the same image.

Your task: for each entity in the pre-extracted list, determine whether the model-generated HTML contains that entity.

# Two-level judgment per entity
For each entity output TWO boolean fields:
- "found_in_prediction": true if the entity's value appears anywhere in the model HTML in a
  recognizable form — even if slightly corrupted (e.g., missing comma, merged spaces, truncated).
  Set to false only if the entity is completely absent from the prediction.
- "correct": true ONLY if the entity satisfies the strict matching rules below.
  (correct=true implies found_in_prediction=true; found_in_prediction=true does NOT imply correct=true)

# PRIORITY RULE: exact match → always correct
If the matched_text (whitespace-normalized, case-insensitive) is identical to value, set correct=true.
The context_hint is only used to DISAMBIGUATE when the same value string appears in multiple different
locations in the prediction. It is NOT a veto: a different surrounding context does NOT make an
otherwise exact match incorrect.

# Context-hint matching (apply loosely, only for disambiguation)
The context_hint is extracted from annotated HTML and may have minor formatting differences vs the prediction:
- Spaces may appear around currency symbols in the hint (e.g., "$ 66.59") but not in the prediction ("$66.59").
- Commas around numbers may differ in spacing (e.g., "per share, inclusive" vs "per share,inclusive").
- Apply whitespace-normalization and ignore spacing around $, %, (, ) when comparing context.
- The entity value itself (e.g., "66.59") must still appear in or very near the matched context region.
- IMPORTANT: Do NOT reject a match just because the context_hint has different spacing than the prediction.
  If the surrounding words match (e.g., "Purchase prices range from", "per share, inclusive"), that is
  sufficient context confirmation — minor punctuation/spacing differences are expected and acceptable.

# Strict matching rules (apply when setting "correct")
- For "Number", "Temporal", "Monetary Unit":
  * correct=true if the exact numeric/date string (whitespace-normalized, case-insensitive) appears in
    the model HTML. If it appears multiple times, use context_hint to pick the right occurrence.
  * Do NOT mark correct if the string is corrupted, truncated, or merged with adjacent characters.
    Examples of NOT correct: "ERIBOURGPAUL" for "FRIBOURG PAUL J"; "25700" for "25,700"; "0.5" for "0.50".
  * Mark found_in_prediction=true if a close but not exact version appears (e.g., "25700" for "25,700").
- For "Reporting Entity", "Financial Concepts":
  * correct=true if the full string appears (whitespace-normalized, case-insensitive) in the model HTML.
    If it appears multiple times, use context_hint to pick the right occurrence.
  * Minor run-together spaces within a single token (e.g., "UNITEDSTATES" for "UNITED STATES") may be
    marked correct if still fully recognizable — record actual text in matched_text.
  * Missing words, swapped words, or substituted characters → correct=false, but may be found_in_prediction=true.

# Step 1: Match entities
For each entity in the pre-extracted list, output a diagnostic record with:
  - "type", "value", "context_hint" (copied from input)
  - "found_in_prediction": true/false (recognizable attempt exists in prediction)
  - "correct": true/false (exact match per strict rules above; exact match always overrides context check)
  - "matched_text": the actual text found in the model HTML (empty string "" if not found at all)

# Step 2: Compute counts
The total_entities counts come directly from the pre-extracted list — do NOT re-count from the ground truth HTML.
Compute:
  - found_entities = number of diagnostics where found_in_prediction = true
  - correct_entities = number of diagnostics where correct = true
  - found_entities_with_X_type and correct_entities_with_X_type for each of the 5 types
  - entity_accuracy = correct_entities / found_entities * 100, rounded to 2 decimal places
    (0.0 if found_entities = 0)

# Step 3: Output
Output exactly one valid JSON object:
{
  "total_entities": <int>,
  "total_entities_with_Number_type": <int>,
  "total_entities_with_Temporal_type": <int>,
  "total_entities_with_Monetary_Unit_type": <int>,
  "total_entities_with_Reporting_Entity_type": <int>,
  "total_entities_with_Financial_Concepts_type": <int>,
  "found_entities": <int>,
  "found_entities_with_Number_type": <int>,
  "found_entities_with_Temporal_type": <int>,
  "found_entities_with_Monetary_Unit_type": <int>,
  "found_entities_with_Reporting_Entity_type": <int>,
  "found_entities_with_Financial_Concepts_type": <int>,
  "correct_entities": <int>,
  "correct_entities_with_Number_type": <int>,
  "correct_entities_with_Temporal_type": <int>,
  "correct_entities_with_Monetary_Unit_type": <int>,
  "correct_entities_with_Reporting_Entity_type": <int>,
  "correct_entities_with_Financial_Concepts_type": <int>,
  "entity_accuracy": <float 0-100>,
  "overall_explanation": "<1-2 sentences summarizing model quality>",
  "entity_diagnostics": [
    {"type": "...", "value": "...", "context_hint": "...",
     "found_in_prediction": true, "correct": false, "matched_text": "25700"},
    ...
  ]
}

Rules:
* Output only valid JSON (no extra text).
* All count fields must be integers, not strings.
* entity_accuracy must be a float (e.g. 85.71), NOT a string like "85.71%".
* entity_accuracy = correct_entities / found_entities * 100 (NOT correct / total).
* total_entities counts MUST match the counts from the pre-extracted list.
* REMINDER: if matched_text equals value (normalized), correct MUST be true — do not set correct=false
  just because the surrounding context in the prediction differs from context_hint.
"""

In [64]:
def get_user_prompt(gold_entities: list, gold_html_stripped: str, model_html: str) -> str:
    """Build the user-turn message for the LLM judge.

    Combines three inputs into a single prompt string:
        1. gold_entities  — the pre-extracted entity list (type, value, context_hint)
                            This is what the judge iterates over; counts come from here.
        2. gold_html_stripped — gold HTML with entity tags removed, for structural context.
        3. model_html     — the OCR model's predicted HTML output (no entity tags).

    The judge is instructed not to re-count entities from the HTML — it must use
    the pre-extracted list as the authoritative source of truth.
    """
    entities_json = json.dumps(gold_entities, indent=2)
    return f"""# Inputs

## Pre-extracted Gold Entities (authoritative — do not re-count from ground truth HTML)
{entities_json}

## Ground Truth HTML (entity tags stripped, for positional context only)
{gold_html_stripped}

## Model-Generated HTML (no entity tags)
{model_html}

# Output
"""

In [65]:
def get_response(user_input):
    """Send the judge prompt to GPT-4o and return the raw JSON string response.

    Uses:
        - response_format="json_object" to guarantee valid JSON output.
        - temperature=0 for near-deterministic results across runs.
          (LLM judges are stochastic by default; temperature=0 greatly reduces
          run-to-run variance, which is important for reproducible evaluation.)
    """
    completion = Client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_PROMPT_TEMPLATE},
            {"role": "user",   "content": user_input}
        ]
    )
    return completion.choices[0].message.content

In [66]:
def read_html_txt(file_path):
    """Read an HTML/text file, falling back to latin-1 if the file is not valid UTF-8.

    Financial document HTMLs occasionally contain non-UTF-8 bytes (e.g., from
    legacy Windows-encoded SEC filings), so we try UTF-8 first and fall back
    to latin-1 rather than crashing.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
    except UnicodeDecodeError:
        with open(file_path, "r", encoding="latin-1") as f:
            content = f.read()
    return content

In [67]:
def write_case_result(output_path: str, result: list):
    """Serialize the Results list to a pretty-printed JSON file.

    Creates parent directories if they don't exist.
    Each element in `result` is one sample's evaluation dict, containing
    entity counts, accuracy, diagnostics, and the sample id.
    """
    parent_dir = os.path.dirname(output_path)
    os.makedirs(parent_dir, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4)

    print(f"Results saved to {output_path}")

In [68]:
def evaluate_performance(result: list):
    """Aggregate entity-level evaluation metrics across all evaluated samples.

    Metric design — precision-style accuracy:
        accuracy = correct_entities / found_entities  (NOT correct / total_gold)

    This means: of the entities the model attempted to extract (found in the
    prediction in any form), what fraction did it get exactly right?
    Entities the model missed entirely (not found at all) are excluded from the
    denominator — they affect recall, not precision.

    Example: gold has 7 monetary entities; model produced 5, of which 4 are
    exact. monetary_unit_entity_acc = 4/5 = 80%, not 4/7 = 57%.

    Args:
        result: list of per-sample dicts as produced by the main evaluation loop.
                Each dict must contain found_entities_with_X_type and
                correct_entities_with_X_type fields for all 5 entity types.

    Returns:
        dict with overall and per-type accuracy percentages (0.0–100.0).
        Returns 0.0 for any type where found == 0 (avoids division by zero).
    """
    # Maps short key → (found field name, correct field name) in each result dict
    key_map = {
        "number":             ("found_entities_with_Number_type",             "correct_entities_with_Number_type"),
        "temporal":           ("found_entities_with_Temporal_type",           "correct_entities_with_Temporal_type"),
        "monetary_unit":      ("found_entities_with_Monetary_Unit_type",      "correct_entities_with_Monetary_Unit_type"),
        "reporting_entity":   ("found_entities_with_Reporting_Entity_type",   "correct_entities_with_Reporting_Entity_type"),
        "financial_concepts": ("found_entities_with_Financial_Concepts_type", "correct_entities_with_Financial_Concepts_type"),
    }

    total_found   = 0
    total_correct = 0
    founds   = {k: 0 for k in key_map}
    corrects = {k: 0 for k in key_map}

    # Accumulate counts across all samples
    for line in result:
        total_found   += line.get("found_entities", 0)
        total_correct += line.get("correct_entities", 0)
        for k, (fnd_key, cor_key) in key_map.items():
            founds[k]   += line.get(fnd_key, 0)
            corrects[k] += line.get(cor_key, 0)

    def acc(c, t):
        return round(c / t * 100, 2) if t else 0.0

    return {
        "entity_acc":                    acc(total_correct, total_found),
        "number_entity_acc":             acc(corrects["number"],             founds["number"]),
        "temporal_entity_acc":           acc(corrects["temporal"],           founds["temporal"]),
        "monetary_unit_entity_acc":      acc(corrects["monetary_unit"],      founds["monetary_unit"]),
        "reporting_entity_acc":          acc(corrects["reporting_entity"],   founds["reporting_entity"]),
        "financial_concepts_entity_acc": acc(corrects["financial_concepts"], founds["financial_concepts"]),
    }

In [ ]:
# ── Main evaluation loop ──────────────────────────────────────
# Processes samples [start_idx, end_idx) and appends each result to Results.
# Each iteration:
#   1. Load gold + prediction HTML for sample i.
#   2. Deterministically extract entities from gold (BeautifulSoup, no LLM).
#   3. Strip structural HTML tags from the prediction so the judge receives
#      plain text regardless of model type (PaddleOCR = plain text already;
#      LLMs like Llama/GPT-4o output full HTML which we flatten here).
#   4. Call GPT-4o judge to assess found_in_prediction / correct per entity.
#   5. Overwrite entity type counts using authoritative programmatic values
#      (LLM's self-reported counts are unreliable).
#   6. Apply deterministic override: matched_text == value → correct=True
#      (guards against LLM incorrectly failing exact string matches).
#   7. Recompute found_* and correct_* counts from the (now-corrected) diagnostics.

Results = []
for i in tqdm(range(start_idx, end_idx)):
    GROUND_TRUTH_HTML_PATH = os.path.join(gold_html, f"gold_{i}.txt")
    MODEL_HTML_PATH        = os.path.join(model_html, f"pred_{i}.txt")

    GROUND_TRUTH_HTML = read_html_txt(GROUND_TRUTH_HTML_PATH)
    MODEL_HTML        = read_html_txt(MODEL_HTML_PATH)

    # Step 1: Extract entities from gold HTML using BeautifulSoup (deterministic, no LLM)
    gold_entities      = extract_gold_entities(GROUND_TRUTH_HTML)
    gold_html_stripped = strip_entity_tags(GROUND_TRUTH_HTML)

    # Step 2: Flatten model prediction to plain text.
    # LLMs output HTML (<table>, <td>, etc.); OCR tools output plain text.
    # Stripping tags normalizes both to the same format for the judge.
    model_text = strip_html_tags(MODEL_HTML)

    # Step 3: Ask the LLM judge to evaluate each entity in the prediction
    user_input = get_user_prompt(gold_entities, gold_html_stripped, model_text)
    response   = get_response(user_input)
    result     = json.loads(response)
    result["id"] = i  # track which sample this result belongs to

    # Step 4: Overwrite total_entities_with_X counts using programmatic values.
    # The LLM may miscount; BeautifulSoup is authoritative for totals.
    type_counts = {}
    for e in gold_entities:
        type_counts[e["type"]] = type_counts.get(e["type"], 0) + 1
    result["total_entities"]                             = len(gold_entities)
    result["total_entities_with_Number_type"]            = type_counts.get("Number", 0)
    result["total_entities_with_Temporal_type"]          = type_counts.get("Temporal", 0)
    result["total_entities_with_Monetary_Unit_type"]     = type_counts.get("Monetary Unit", 0)
    result["total_entities_with_Reporting_Entity_type"]  = type_counts.get("Reporting Entity", 0)
    result["total_entities_with_Financial_Concepts_type"]= type_counts.get("Financial Concepts", 0)

    # Step 5: Deterministic override for exact matches.
    # If the LLM's matched_text equals the gold value (whitespace-normalized,
    # case-insensitive), force correct=True. This prevents the LLM from failing
    # an exact match just because the surrounding context_hint differs slightly.
    _norm = lambda s: re.sub(r'\s+', ' ', s).strip().lower()
    for d in result.get("entity_diagnostics", []):
        mt  = _norm(d.get("matched_text", ""))
        val = _norm(d.get("value", ""))
        if mt and mt == val:
            d["correct"] = True

    # Step 6: Recompute found_* and correct_* counts from diagnostics.
    # These are more reliable than what the LLM computed itself.
    found_counts   = {}
    correct_counts = {}
    for d in result.get("entity_diagnostics", []):
        t = d.get("type", "")
        if d.get("found_in_prediction"):
            found_counts[t]   = found_counts.get(t, 0) + 1
        if d.get("correct"):
            correct_counts[t] = correct_counts.get(t, 0) + 1

    result["found_entities"]                              = sum(found_counts.values())
    result["found_entities_with_Number_type"]             = found_counts.get("Number", 0)
    result["found_entities_with_Temporal_type"]           = found_counts.get("Temporal", 0)
    result["found_entities_with_Monetary_Unit_type"]      = found_counts.get("Monetary Unit", 0)
    result["found_entities_with_Reporting_Entity_type"]   = found_counts.get("Reporting Entity", 0)
    result["found_entities_with_Financial_Concepts_type"] = found_counts.get("Financial Concepts", 0)

    result["correct_entities"]                              = sum(correct_counts.values())
    result["correct_entities_with_Number_type"]             = correct_counts.get("Number", 0)
    result["correct_entities_with_Temporal_type"]           = correct_counts.get("Temporal", 0)
    result["correct_entities_with_Monetary_Unit_type"]      = correct_counts.get("Monetary Unit", 0)
    result["correct_entities_with_Reporting_Entity_type"]   = correct_counts.get("Reporting Entity", 0)
    result["correct_entities_with_Financial_Concepts_type"] = correct_counts.get("Financial Concepts", 0)

    Results.append(result)

In [73]:
# Inspect the first result in the list.
# Each result dict contains:
#   - id: sample index (maps to gold_{id}.txt / pred_{id}.txt)
#   - total_entities_with_X_type: gold entity counts (from BeautifulSoup)
#   - found_entities_with_X_type: how many gold entities appear in the prediction
#   - correct_entities_with_X_type: how many of those are exact matches
#   - entity_accuracy: correct / found * 100 (precision-style)
#   - entity_diagnostics: per-entity breakdown with found_in_prediction, correct, matched_text
Results[0]

{'total_entities': 1,
 'total_entities_with_Number_type': 0,
 'total_entities_with_Temporal_type': 0,
 'total_entities_with_Monetary_Unit_type': 0,
 'total_entities_with_Reporting_Entity_type': 1,
 'total_entities_with_Financial_Concepts_type': 0,
 'found_entities': 1,
 'found_entities_with_Number_type': 0,
 'found_entities_with_Temporal_type': 0,
 'found_entities_with_Monetary_Unit_type': 0,
 'found_entities_with_Reporting_Entity_type': 1,
 'found_entities_with_Financial_Concepts_type': 0,
 'correct_entities': 1,
 'correct_entities_with_Number_type': 0,
 'correct_entities_with_Temporal_type': 0,
 'correct_entities_with_Monetary_Unit_type': 0,
 'correct_entities_with_Reporting_Entity_type': 1,
 'correct_entities_with_Financial_Concepts_type': 0,
 'entity_accuracy': 100.0,
 'overall_explanation': "The model successfully identified and correctly matched the single reporting entity 'AENB' in the document.",
 'entity_diagnostics': [{'type': 'Reporting Entity',
   'value': 'AENB',
   'conte

In [71]:
# Save all results to a JSON file for offline analysis and reproducibility.
# File is named after model_name_test so different runs don't overwrite each other.
write_case_result(os.path.join(case_result, f"{model_name_test}.json"), Results)

Results saved to results/MM_2026_Results/llm_as_judge_output/paddleocrv5_zero-shot_test.json


In [72]:
# Compute aggregated accuracy across all evaluated samples.
# Returns precision-style accuracy per entity type:
#   entity_acc                    — overall (all types combined)
#   number_entity_acc             — numeric values (e.g., "25,700")
#   temporal_entity_acc           — dates/years (e.g., "02/05/2025")
#   monetary_unit_entity_acc      — monetary values (e.g., "$65.63", "Dollars in Thousands")
#   reporting_entity_acc          — org/person names (e.g., "American Express National Bank")
#   financial_concepts_entity_acc — financial terms (e.g., "Principal amount")
evaluate_performance(Results)

{'entity_acc': 84.85,
 'number_entity_acc': 100.0,
 'temporal_entity_acc': 100.0,
 'monetary_unit_entity_acc': 83.33,
 'reporting_entity_acc': 78.95,
 'financial_concepts_entity_acc': 0.0}